# E4 - Concellos da provincia da Coruña

Extraer los datos de población de los 93 municipios de la
provincia de A Coruña desde Wikipedia mediante web scraping.

Después limpiaremos y validaremos los datos para poder
analizarlos y responder las preguntas del ejercicio.

## 4.1. Antes de descargar

Antes de realizar el web scraping, compruebo las reglas del archivo robots.txt de Wikipedia para saber si permiten acceder de forma automatizada a la página que quiero consultar.

In [2]:
from urllib.robotparser import RobotFileParser

# Guardamos las direcciones que vamos a utilizar
robots_url = "https://en.wikipedia.org/robots.txt"
pagina_url = "https://en.wikipedia.org/wiki/List_of_municipalities_in_A_Coru%C3%B1a"

In [3]:
# Creamos el objeto para leer las reglas de robots.txt
robot = RobotFileParser()

# Le indicamos dónde está el archivo robots.txt
robot.set_url(robots_url)

# Leemos las reglas de Wikipedia
robot.read()

In [ ]:
# comprobamos si podemos acceder a la página de los municipios
permitido = robot.can_fetch("*", pagina_url)

# Mostramos el resultado
print("¿Podemos acceder a la página?", permitido)

¿Podemos acceder a la página? False


In [ ]:
# comprobamos si se ha podido leer el archivo robots.txt
print("Código de respuesta:", robot.mtime())
print("¿Se ha bloqueado el acceso?", robot.disallow_all)
print("¿Se permite todo el acceso?", robot.allow_all)

In [5]:
print("Momento de lectura:", robot.mtime())

Momento de lectura: 0


In [7]:
# comprobamos si se han podido leer las reglas
print("Momento de lectura:", robot.mtime())
print("Acceso bloqueado:", robot.disallow_all)
print("Acceso permitido:", robot.allow_all)

Momento de lectura: 0
Acceso bloqueado: True
Acceso permitido: False


In [ ]:
from urllib.request import urlopen
from urllib.error import HTTPError, URLError

# intentamos consultar robots.txt para ver qué responde Wikipedia
try:
    respuesta = urlopen(robots_url, timeout=10)
    print("Código HTTP:", respuesta.status)
    respuesta.close()

except HTTPError as error:
    print("Error HTTP:", error.code)

except URLError as error:
    print("Error de conexión:", error.reason)

Error HTTP: 403


In [ ]:
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

# identificamos nuestro proyecto al hacer la petición
agente = "E4Concellos/1.0 (contacto: tu-correo@example.com)"

# Preparamos la petición con nuestro User-Agent
peticion = Request(robots_url, headers={"User-Agent": agente})

try:
    respuesta = urlopen(peticion, timeout=10)
    print("Código HTTP:", respuesta.status)
    respuesta.close()

except HTTPError as error:
    print("Error HTTP:", error.code)

except URLError as error:
    print("Error de conexión:", error.reason)

Código HTTP: 200


In [18]:
# creamos un nuevo objeto para leer las reglas
robot = RobotFileParser()

# descargamos robots.txt usando nuestro User-Agent
respuesta = urlopen(peticion, timeout=10)
contenido = respuesta.read().decode("utf-8")
respuesta.close()

# pasamos las reglas descargadas 
robot.parse(contenido.splitlines())

# comprobamos si podemos acceder a la página
permitido = robot.can_fetch(agente, pagina_url)

print("¿se puede acceder a la página?", permitido)

¿se puede acceder a la página? True


### Licencia y datos personales

Wikipedia utiliza la licencia CC BY-SA 4.0, que permite
copiar, compartir y modificar su contenido, siempre que
se indique la fuente, se reconozca la autoría y se respete
la condición de compartir las adaptaciones bajo la misma
licencia o una compatible.

En este ejercicio vamos a utilizar los datos de población
de los municipios de A Coruña.

La tabla no contiene datos personales, ya que muestra
nombres de municipios y cifras de población, sin
identificar a personas concretas.

Fuente: Wikipedia.
Licencia: CC BY-SA 4.0.

## 4.2. Descarga

Vamos a descargar la página de Wikipedia con la librería
requests y guardar el HTML original en la carpeta datos/raw/.

Antes de descargar, comprobaremos si el archivo ya existe
para no repetir la descarga cada vez que ejecutemos el notebook.

In [12]:
import requests
from pathlib import Path


/Users/paulavargas/e4-concellos/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [14]:
# indicamos dónde vamos a guardar el HTML
carpeta_raw = Path("../datos/raw")
archivo_html = carpeta_raw / "municipios_coruna.html"

print("carpeta actual:", Path.cwd())
print("archivo HTML:", archivo_html)

# creamos la carpeta si todavía no existe
carpeta_raw.mkdir(parents=True, exist_ok=True)

print("¿existe la carpeta?", carpeta_raw.exists())
print("¿existe el HTML?", archivo_html.exists())

carpeta actual: /Users/paulavargas/e4-concellos/E4_concellos/notebooks
archivo HTML: ../datos/raw/municipios_coruna.html
¿existe la carpeta? True
¿existe el HTML? False


In [16]:
# identificamos nuestro proyecto al hacer la petición
headers = {
    "User-Agent": "E4Concellos/1.0 (contacto: paulavargasilva09@gmail.com)"
}

In [19]:
# comprobamos si ya tenemos el archivo descargado
if archivo_html.exists():
    print("El archivo ya existe, no hace falta descargarlo.")
else:
    # descargamos la página de Wikipedia
    respuesta = requests.get(pagina_url, headers=headers, timeout=15)
    respuesta.raise_for_status()

    # guardamos el HTML original
    archivo_html.write_bytes(respuesta.content)

    print("Página descargada y guardada correctamente.")

El archivo ya existe, no hace falta descargarlo.


## 4.3. Extracción de datos

Voy a leer el HTML que he guardado y buscar la tabla que contiene los municipios y su población.

In [20]:
from bs4 import BeautifulSoup

In [21]:
# leemos el HTML que hemos guardado
html = archivo_html.read_bytes()

# convertimos el HTML en un objeto que podamos analizar
sopa = BeautifulSoup(html, "html.parser")

print("HTML cargado correctamente")

HTML cargado correctamente


In [22]:
# buscamos todas las tablas del HTML
tablas = sopa.find_all("table")

# mostramos cuántas hemos encontrado
print("Número de tablas encontradas:", len(tablas))

Número de tablas encontradas: 2


In [23]:
# mostramos los encabezados de cada tabla
for numero, tabla in enumerate(tablas, start=1):
    encabezados = tabla.find_all("th")

    print("Tabla", numero)

    for encabezado in encabezados[:10]:
        print(encabezado.get_text(strip=True))

    print("--------------------")

Tabla 1
Name
Population (2024)[20]
--------------------
Tabla 2
vteMunicipalities of Spain
Byautonomous community
Byprovince
--------------------


In [25]:
# buscamos la tabla por sus encabezados
tabla_municipios = None

for tabla in tablas:
    encabezados = tabla.find_all("th")
    textos = [th.get_text(" ", strip=True) for th in encabezados]

    if "Name" in textos and any("Population" in texto for texto in textos):
        tabla_municipios = tabla
        break

print("¿ encontramos la tabla?", tabla_municipios is not None)

¿ encontramos la tabla? True


In [26]:
# buscamos todas las filas de la tabla
filas = tabla_municipios.find_all("tr")

# mostramos cuántas filas hay
print("Número de filas:", len(filas))

# mostramos el contenido de las primeras 3 filas
for fila in filas[:3]:
    print(fila.get_text(" | ", strip=True))

Número de filas: 94
Name | Population (2024) | [ | 20 | ]
Abegondo | 5,578
Ames | 32,812


In [27]:
# creamos una lista para guardar los datos
datos_municipios = []

# recorremos las filas sin contar la cabecera
for fila in filas[1:]:
    columnas = fila.find_all("td")

    if len(columnas) >= 2:
        nombre = columnas[0].get_text(" ", strip=True)
        poblacion = columnas[1].get_text(" ", strip=True)

        datos_municipios.append([nombre, poblacion])

# mostramos los primeros 5 municipios
print(datos_municipios[:5])

[['Abegondo', '5,578'], ['Ames', '32,812'], ['Aranga', '1,799'], ['Ares', '6,156'], ['Arteixo', '34,038']]


In [28]:
# importamos pandas
import pandas as pd

# creamos una tabla con los datos extraídos
df = pd.DataFrame(datos_municipios, columns=["Municipio", "Poblacion"])

# mostramos las primeras filas
df.head()

,Municipio,Poblacion
0,Abegondo,"5,578"
1,Ames,"32,812"
2,Aranga,"1,799"
3,Ares,"6,156"
4,Arteixo,"34,038"


## 4.4. Limpieza de datos

Voy a revisar cómo están guardadas las poblaciones y convertirlas en números enteros para poder hacer cálculos.


In [29]:
# contamos los municipios que hemos extraído
print("Número de municipios:", len(df))

# comprobamos el tamaño de la tabla
print("Filas y columnas:", df.shape)

Número de municipios: 93
Filas y columnas: (93, 2)


In [30]:
# comprobamos el tipo de datos de cada columna
print(df.dtypes)

# mostramos algunas poblaciones
print(df["Poblacion"].head(10))

Municipio    object
Poblacion    object
dtype: object
0     5,578
1    32,812
2     1,799
3     6,156
4    34,038
5     5,887
6     3,291
7     6,986
8    13,261
9     1,801
Name: Poblacion, dtype: object


In [31]:
# buscamos poblaciones que tengan caracteres diferentes de números y comas
valores_raros = df[
    ~df["Poblacion"].str.fullmatch(r"[0-9,]+")
]

print("Valores con formato diferente:", len(valores_raros))
print(valores_raros)

Valores con formato diferente: 0
Empty DataFrame
Columns: [Municipio, Poblacion]
Index: []


In [32]:
# comprobamos que las comas estén bien colocadas
formato_incorrecto = df[
    ~df["Poblacion"].str.fullmatch(r"\d{1,3}(,\d{3})*")
]

print("Poblaciones con formato incorrecto:", len(formato_incorrecto))
print(formato_incorrecto)

Poblaciones con formato incorrecto: 0
Empty DataFrame
Columns: [Municipio, Poblacion]
Index: []


In [33]:
# quitamos las comas y convertimos las poblaciones a números enteros
df["Poblacion"] = df["Poblacion"].str.replace(",", "", regex=False).astype(int)

# comprobamos el resultado
print(df.dtypes)
df.head()

Municipio    object
Poblacion     int64
dtype: object


,Municipio,Poblacion
0,Abegondo,5578
1,Ames,32812
2,Aranga,1799
3,Ares,6156
4,Arteixo,34038


### Resultado de la limpieza

Al revisar los datos, vi que las poblaciones estaban guardadas como texto y utilizaban comas para separar los millares.

Comprobé que todas tenían el formato correcto y no encontré valores extraños, así que no tuve que corregir ninguno.

Después quité las comas y convertí las poblaciones en números enteros para poder hacer los cálculos del ejercicio.

## 4.5. Validación de los datos

Voy a comprobar que tengo los 93 municipios, que no hay nombres repetidos y que las poblaciones coinciden con los datos de la página.

In [34]:
# comprobamos que tenemos 93 municipios
print("Número de municipios:", len(df))
print("¿Tenemos 93 municipios?", len(df) == 93)

# buscamos municipios con el nombre repetido
duplicados = df[df["Municipio"].duplicated()]

print("Municipios duplicados:", len(duplicados))
print(duplicados)

Número de municipios: 93
¿Tenemos 93 municipios? True
Municipios duplicados: 0
Empty DataFrame
Columns: [Municipio, Poblacion]
Index: []


In [35]:
# sumamos la población de todos los municipios
poblacion_total = df["Poblacion"].sum()

print("Población total de los municipios:", poblacion_total)

Población total de los municipios: 1128320


In [36]:
# buscamos en el texto de Wikipedia las líneas que hablan de población
texto_pagina = sopa.get_text("\n", strip=True)

for linea in texto_pagina.splitlines():
    if "population" in linea.lower():
        print(linea)

12th largest by population
Municipalities are categorised by population for the purpose of determining the number of councillors: three when the population is up to 100 inhabitants, five for 101
whilst those with a population lower than 250 use a
, with meetings occurring more or less frequently depending on the population of the municipality: monthly for those whose population is larger than 20,000, once every two months if it ranges between 5,001 and 20,000, and once every three months if it does not exceed 5,000.
The largest municipality by population in the province as of the 2024 Spanish census is
Largest municipalities in the province of A Coruña by population
is the second largest municipality in A Coruña by population
Population (2024)


In [37]:
# dividimos el texto de la página en líneas
lineas = texto_pagina.splitlines()

# buscamos las referencias a la población de la provincia
for i, linea in enumerate(lineas):
    if "12th largest by population" in linea.lower():
        print("\n".join(lineas[max(0, i-5):i+6]))

2024
[
update
]
Spanish census, the province is the
12th largest by population
, with
1,128,320
inhabitants,
[
1


In [40]:
# buscamos la población provincial en el texto de Wikipedia
import re

coincidencia = re.search(
    r"with\s+([\d,]+)\s+inhabitants",
    texto_pagina
)

if coincidencia:
    poblacion_provincia = int(coincidencia.group(1).replace(",", ""))

    print("Población de los municipios:", poblacion_total)
    print("Población de la provincia:", poblacion_provincia)
    print("¿Coinciden?", poblacion_total == poblacion_provincia)
else:
    print("No se encontró la población provincial en el texto.")

Población de los municipios: 1128320
Población de la provincia: 1128320
¿Coinciden? True


In [41]:
# leemos las tablas del HTML con pandas
tablas_pandas = pd.read_html(str(archivo_html))

# comprobamos cuántas tablas encuentra
print("Tablas encontradas por pandas:", len(tablas_pandas))

# mostramos las primeras filas de cada tabla
for numero, tabla in enumerate(tablas_pandas, start=1):
    print("Tabla", numero)
    print(tabla.head(3))
    print("--------------------")

Tablas encontradas por pandas: 2
Tabla 1
       Name  Population (2024)[20]
0  Abegondo                   5578
1      Ames                  32812
2    Aranga                   1799
--------------------
Tabla 2
        vteMunicipalities of Spain  \
0  List of municipalities of Spain   
1          By autonomous community   
2                      By province   

                        vteMunicipalities of Spain.1  
0                    List of municipalities of Spain  
1  Andalusia Aragon Asturias1 Balearic Islands1 B...  
2  A Coruña Álava Albacete Alicante Almería Astur...  
--------------------


In [42]:
# seleccionamos la tabla de municipios de pandas
df_pandas = tablas_pandas[0].copy()

# ponemos los mismos nombres de columnas que en nuestro DataFrame
df_pandas.columns = ["Municipio", "Poblacion"]

# comparamos los datos de las dos tablas
coinciden = df.equals(df_pandas)

print("¿Coinciden los datos?", coinciden)
print("Municipios encontrados por pandas:", len(df_pandas))

¿Coinciden los datos? True
Municipios encontrados por pandas: 93


### Resultado de la validación

He comprobado que tengo los 93 municipios de la provincia y que no hay nombres repetidos.

La suma de las poblaciones es de 1.128.320 habitantes, que coincide con el total indicado en Wikipedia.

También he comparado los datos que he extraído con BeautifulSoup con los obtenidos mediante pd.read_html() y coinciden.

## 4.6. Preguntas sobre los datos

En este apartado voy a utilizar los datos que he obtenido para responder a las preguntas del ejercicio.

### 1. Los cinco municipios con más población

In [43]:
# ordeno los municipios de mayor a menor población
top_5 = df.sort_values("Poblacion", ascending=False).head(5)

# muestro los cinco municipios más poblados
top_5

,Municipio,Poblacion
29,A Coruña,249261
77,Santiago de Compostela,99536
35,Ferrol,64218
53,Narón,39285
57,Oleiros,38333


**Resultado:**
Los cinco municipios con más población son A Coruña, Santiago de Compostela, Ferrol, Narón y Oleiros. A Coruña es el más poblado, con 249.261 habitantes, bastante por encima del resto.

### 2. Municipios con menos de 5.000 habitantes

In [44]:
# selecciono los municipios con menos de 5000 habitantes
municipios_pequenos = df[df["Poblacion"] < 5000]

# cuento cuántos municipios cumplen la condición
cantidad = len(municipios_pequenos)

# calculo qué porcentaje representan
porcentaje = cantidad / len(df) * 100

print("Municipios con menos de 5000 habitantes:", cantidad)
print("Porcentaje:", round(porcentaje, 2), "%")

Municipios con menos de 5000 habitantes: 43
Porcentaje: 46.24 %


**Resultado:**
He comprobado que 43 de los 93 municipios tienen menos de 5.000 habitantes. Esto representa el 46,24 % del total, es decir, casi la mitad de los municipios de la provincia.

### 3. Porcentaje de población de los tres municipios más poblados

In [45]:
# selecciono los tres municipios más poblados
top_3 = df.sort_values("Poblacion", ascending=False).head(3)

# sumo sus poblaciones
poblacion_top_3 = top_3["Poblacion"].sum()

# calculo el porcentaje respecto al total provincial
porcentaje_top_3 = poblacion_top_3 / poblacion_total * 100

print("Población de los tres municipios:", poblacion_top_3)
print("Porcentaje de la población provincial:", round(porcentaje_top_3, 2), "%")

Población de los tres municipios: 413015
Porcentaje de la población provincial: 36.6 %


**Resultado:**
Los tres municipios más poblados son A Coruña, Santiago de Compostela y Ferrol. Al sumar sus poblaciones obtengo 413.015 habitantes, que representan el 36,6 % de la población total de la provincia.

### 4. Municipios necesarios para alcanzar la mitad de la población

In [46]:
# ordeno los municipios de mayor a menor población
municipios_ordenados = df.sort_values("Poblacion", ascending=False)

# calculo la mitad de la población provincial
mitad_poblacion = poblacion_total / 2

# voy sumando las poblaciones de los municipios
poblacion_acumulada = 0
cantidad_municipios = 0

for poblacion in municipios_ordenados["Poblacion"]:
    poblacion_acumulada += poblacion
    cantidad_municipios += 1

    if poblacion_acumulada >= mitad_poblacion:
        break

print("Mitad de la población provincial:", mitad_poblacion)
print("Municipios necesarios:", cantidad_municipios)
print("Población acumulada:", poblacion_acumulada)

Mitad de la población provincial: 564160.0
Municipios necesarios: 8
Población acumulada: 589078


**Resultado:**
He comprobado que necesito sumar los 8 municipios más poblados para alcanzar al menos la mitad de la población provincial. Entre ellos reúnen 589.078 habitantes, superando los 564.160 que representan el 50 % del total.

## 4.7. Guardar los datos

Voy a guardar los datos limpios en un archivo CSV dentro de la carpeta `datos/clean/`, utilizando la codificación UTF-8.

In [47]:
# indico dónde voy a guardar el archivo CSV
carpeta_clean = Path("../datos/clean")
archivo_csv = carpeta_clean / "municipios_coruna.csv"

# creo la carpeta si todavía no existe
carpeta_clean.mkdir(parents=True, exist_ok=True)

# guardo los datos limpios en formato CSV
df.to_csv(archivo_csv, index=False, encoding="utf-8")

print("Archivo CSV guardado correctamente.")
print("Ruta:", archivo_csv)

Archivo CSV guardado correctamente.
Ruta: ../datos/clean/municipios_coruna.csv


In [48]:
# leo el archivo CSV que acabo de guardar
df_csv = pd.read_csv(archivo_csv, encoding="utf-8")

# compruebo que tiene los datos esperados
print("Municipios guardados:", len(df_csv))
print("¿Coinciden los datos?", df.equals(df_csv))

# muestro las primeras filas
df_csv.head()

Municipios guardados: 93
¿Coinciden los datos? True


,Municipio,Poblacion
0,Abegondo,5578
1,Ames,32812
2,Aranga,1799
3,Ares,6156
4,Arteixo,34038


### Resultado
He guardado los datos limpios en el archivo municipios_coruna.csv, dentro de la carpeta datos/clean/.

Después he vuelto a leer el archivo con pandas y he comprobado que contiene los 93 municipios y que los datos coinciden con los originales.

## Uso de inteligencia artificial

Para realizar este ejercicio he utilizado herramientas de inteligencia artificial como apoyo.

Las he utilizado para ayudarme a escribir y desarrollar el código Python, entender el funcionamiento de las librerías utilizadas y resolver los errores que me han ido apareciendo.

He ejecutado el código en VS Code y he comprobado los resultados obtenidos en cada apartado.